<a href="https://colab.research.google.com/github/username/repo/blob/main/form_to_results_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Form-to-Results Analysis with Gradient Boosting

This notebook replicates the functionality of the Orange3 form-to-results workflow using Python libraries compatible with Google Colab.

In [ ]:
# Install required packages if needed
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, classification_report
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# For Google Colab, set up plotting
plt.style.use('seaborn-v0_8')
%matplotlib inline

## Data Loading and Preparation

In a real scenario, you would upload your form and results data files. For this example, we'll create synthetic data that mimics the structure of horse racing data.

In [ ]:
# Create synthetic form data (similar to what would be in form-data CSV)
np.random.seed(42)
n_samples = 1000

form_data = pd.DataFrame({
    'trainer_id': np.random.choice(range(50), n_samples),
    'jockey_id': np.random.choice(range(100), n_samples),
    'horse_age': np.random.randint(2, 10, n_samples),
    'horse_sex': np.random.choice(['M', 'F', 'G'], n_samples),
    'weight': np.random.uniform(45, 65, n_samples),
    'barrier': np.random.randint(1, 15, n_samples),
    'recent_form_score': np.random.normal(50, 15, n_samples),
    'trainer_win_rate': np.random.beta(2, 3, n_samples),
    'jockey_win_rate': np.random.beta(2, 3, n_samples),
    'days_since_last_race': np.random.exponential(14, n_samples),
    'track_experience': np.random.poisson(5, n_samples),
    'distance_experience': np.random.poisson(3, n_samples),
})

In [ ]:
# Create synthetic results data (similar to what would be in results-data CSV)
# The position is influenced by the form data features
position_noise = np.random.normal(0, 1, n_samples)
position_base = (
    5 - 
    0.3 * form_data['trainer_win_rate'] - 
    0.4 * form_data['jockey_win_rate'] + 
    0.02 * form_data['recent_form_score'] - 
    0.05 * form_data['weight'] + 
    0.1 * form_data['barrier'] + 
    position_noise
)

# Ensure positions are positive integers
position = np.clip(np.round(position_base), 1, 20).astype(int)

results_data = pd.DataFrame({
    'horse_id': range(n_samples),
    'position': position,
    'race_time': np.random.normal(60, 5, n_samples),  # seconds for 1000m race
    'margin': np.maximum(0, np.random.exponential(2, n_samples)),
})

In [ ]:
# Display data shapes and first few rows
print("Form data shape:", form_data.shape)
print("Results data shape:", results_data.shape)
print("\nForm data columns:", list(form_data.columns))
print("\nResults data columns:", list(results_data.columns))

print("\nForm data head:")
print(form_data.head())
print("\nResults data head:")
print(results_data.head())

## Data Merging and Feature Engineering

Replicating the merge and formula operations from the Orange3 workflow

In [ ]:
# Add horse_id to form_data to enable merging
form_data['horse_id'] = range(len(form_data))

# Merge form and results data
merged_data = pd.merge(form_data, results_data, on='horse_id', how='inner')
print(f"Merged data shape: {merged_data.shape}")
print(f"Merged data columns: {list(merged_data.columns)}")

In [ ]:
# Feature engineering - creating additional features
merged_data['combined_skill'] = merged_data['trainer_win_rate'] * merged_data['jockey_win_rate']
merged_data['experience_score'] = (merged_data['track_experience'] + merged_data['distance_experience']) / 2
merged_data['form_efficiency'] = merged_data['recent_form_score'] / merged_data['weight']
merged_data['barrier_advantage'] = 8.5 - abs(merged_data['barrier'] - 8.5)  # Center barrier is often advantageous

# Encode categorical variables
le_sex = LabelEncoder()
merged_data['horse_sex_encoded'] = le_sex.fit_transform(merged_data['horse_sex'])

print("Added engineered features:")
engineered_features = ['combined_skill', 'experience_score', 'form_efficiency', 'barrier_advantage', 'horse_sex_encoded']
print(engineered_features)

In [ ]:
# Select features for modeling
feature_columns = [
    'trainer_id', 'jockey_id', 'horse_age', 'horse_sex_encoded', 'weight',
    'barrier', 'recent_form_score', 'trainer_win_rate', 'jockey_win_rate',
    'days_since_last_race', 'track_experience', 'distance_experience',
    'combined_skill', 'experience_score', 'form_efficiency', 'barrier_advantage'
]

X = merged_data[feature_columns]
y_position = merged_data['position']
y_binary = (merged_data['position'] <= 3).astype(int)  # Top 3 finisher

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y_position.shape}")
print(f"Binary target distribution:\n{y_binary.value_counts()}")

## Model Training and Evaluation

Using Gradient Boosting as in the Orange3 workflow

In [ ]:
# Handle missing values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Standardize features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X_imputed.columns)

print("Data preprocessing completed")

In [ ]:
# Split the data
X_train, X_test, y_pos_train, y_pos_test = train_test_split(
    X_scaled, y_position, test_size=0.2, random_state=42
)

_, _, y_bin_train, y_bin_test = train_test_split(
    X_scaled, y_binary, test_size=0.2, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

In [ ]:
# Train Gradient Boosting Regressor for position prediction
gb_regressor = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

gb_regressor.fit(X_train, y_pos_train)
print("Gradient Boosting Regressor trained")

In [ ]:
# Train Gradient Boosting Classifier for top-3 prediction
gb_classifier = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

gb_classifier.fit(X_train, y_bin_train)
print("Gradient Boosting Classifier trained")

In [ ]:
# Make predictions
y_pos_pred = gb_regressor.predict(X_test)
y_bin_pred = gb_classifier.predict(X_test)
y_bin_pred_proba = gb_classifier.predict_proba(X_test)[:, 1]

# Calculate regression metrics
mse = mean_squared_error(y_pos_test, y_pos_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_pos_test, y_pos_pred)
r2 = r2_score(y_pos_test, y_pos_pred)

# Calculate classification metrics
acc = accuracy_score(y_bin_test, y_bin_pred)

print("REGRESSION METRICS (Position Prediction):")
print(f"MSE: {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R²: {r2:.3f}")

print("\nCLASSIFICATION METRICS (Top-3 Prediction):")
print(f"Accuracy: {acc:.3f}")
print(f"Classification Report:\n{classification_report(y_bin_test, y_bin_pred)}")

## Cross-Validation Analysis

Performing time-series cross-validation to mimic the Orange3 Test & Score functionality

In [ ]:
# Time Series Cross-Validation for regression
tscv = TimeSeriesSplit(n_splits=5)

regression_scores = []
for train_idx, val_idx in tscv.split(X_scaled):
    X_train_cv, X_val_cv = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train_cv, y_val_cv = y_position.iloc[train_idx], y_position.iloc[val_idx]
    
    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42
    )
    model.fit(X_train_cv, y_train_cv)
    y_pred_cv = model.predict(X_val_cv)
    
    mse_cv = mean_squared_error(y_val_cv, y_pred_cv)
    rmse_cv = np.sqrt(mse_cv)
    mae_cv = mean_absolute_error(y_val_cv, y_pred_cv)
    r2_cv = r2_score(y_val_cv, y_pred_cv)
    
    regression_scores.append((mse_cv, rmse_cv, mae_cv, r2_cv))

# Calculate average scores
avg_mse = np.mean([s[0] for s in regression_scores])
avg_rmse = np.mean([s[1] for s in regression_scores])
avg_mae = np.mean([s[2] for s in regression_scores])
avg_r2 = np.mean([s[3] for s in regression_scores])

print("TIME SERIES CROSS-VALIDATION RESULTS (Regression):")
print(f"Average MSE: {avg_mse:.3f}")
print(f"Average RMSE: {avg_rmse:.3f}")
print(f"Average MAE: {avg_mae:.3f}")
print(f"Average R²: {avg_r2:.3f}")

In [ ]:
# Time Series Cross-Validation for classification
classification_scores = []
for train_idx, val_idx in tscv.split(X_scaled):
    X_train_cv, X_val_cv = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train_cv, y_val_cv = y_binary.iloc[train_idx], y_binary.iloc[val_idx]
    
    model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42
    )
    model.fit(X_train_cv, y_train_cv)
    y_pred_cv = model.predict(X_val_cv)
    
    acc_cv = accuracy_score(y_val_cv, y_pred_cv)
    
    classification_scores.append(acc_cv)

avg_acc = np.mean(classification_scores)

print("\nTIME SERIES CROSS-VALIDATION RESULTS (Classification):")
print(f"Average Accuracy: {avg_acc:.3f}")

## Feature Importance Analysis

Analyzing which features are most important for predictions

In [ ]:
# Feature importance for regressor
feature_importance_reg = pd.DataFrame({
    'feature': feature_columns,
    'importance': gb_regressor.feature_importances_
}).sort_values('importance', ascending=False)

# Feature importance for classifier
feature_importance_clf = pd.DataFrame({
    'feature': feature_columns,
    'importance': gb_classifier.feature_importances_
}).sort_values('importance', ascending=False)

print("TOP 10 FEATURES - REGRESSION (Position Prediction):")
print(feature_importance_reg.head(10))

print("\nTOP 10 FEATURES - CLASSIFICATION (Top-3 Prediction):")
print(feature_importance_clf.head(10))

In [ ]:
# Plot feature importances
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Regression feature importance
top_features_reg = feature_importance_reg.head(10)
axes[0].barh(range(len(top_features_reg)), top_features_reg['importance'])
axes[0].set_yticks(range(len(top_features_reg)))
axes[0].set_yticklabels(top_features_reg['feature'])
axes[0].set_xlabel('Importance')
axes[0].set_title('Top Features - Position Prediction')

# Classification feature importance
top_features_clf = feature_importance_clf.head(10)
axes[1].barh(range(len(top_features_clf)), top_features_clf['importance'])
axes[1].set_yticks(range(len(top_features_clf)))
axes[1].set_yticklabels(top_features_clf['feature'])
axes[1].set_xlabel('Importance')
axes[1].set_title('Top Features - Top-3 Prediction')

plt.tight_layout()
plt.show()

## Model Predictions Visualization

Visualizing the model's predictions against actual values

In [ ]:
# Plot actual vs predicted positions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_pos_test, y_pos_pred, alpha=0.6)
plt.plot([y_pos_test.min(), y_pos_test.max()], [y_pos_test.min(), y_pos_test.max()], 'r--', lw=2)
plt.xlabel('Actual Position')
plt.ylabel('Predicted Position')
plt.title(f'Actual vs Predicted Positions (R² = {r2:.3f})')

# Residuals plot
plt.subplot(1, 2, 2)
residuals = y_pos_test - y_pos_pred
plt.scatter(y_pos_pred, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Position')
plt.ylabel('Residuals (Actual - Predicted)')
plt.title('Residuals Plot')

plt.tight_layout()
plt.show()

In [ ]:
# Plot predicted probabilities for top-3 classification
plt.figure(figsize=(10, 6))

# Histogram of predicted probabilities
plt.hist(y_bin_pred_proba[y_bin_test == 0], bins=30, alpha=0.7, label='Not Top 3', density=True)
plt.hist(y_bin_pred_proba[y_bin_test == 1], bins=30, alpha=0.7, label='Top 3', density=True)
plt.xlabel('Predicted Probability of Top 3 Finish')
plt.ylabel('Density')
plt.title('Distribution of Predicted Probabilities by Actual Outcome')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Making Predictions for New Data

Demonstrating how to use the trained models for new race predictions

In [ ]:
# Create sample new data for prediction
new_data = pd.DataFrame({
    'trainer_id': [10, 25, 5],
    'jockey_id': [45, 67, 23],
    'horse_age': [4, 6, 5],
    'horse_sex_encoded': [0, 1, 2],  # M, F, G encoded
    'weight': [55.0, 58.5, 52.3],
    'barrier': [3, 8, 12],
    'recent_form_score': [60.2, 45.8, 52.1],
    'trainer_win_rate': [0.25, 0.35, 0.18],
    'jockey_win_rate': [0.22, 0.31, 0.15],
    'days_since_last_race': [7, 21, 14],
    'track_experience': [8, 12, 5],
    'distance_experience': [4, 6, 3],
    'combined_skill': [0.055, 0.11, 0.027],
    'experience_score': [6.0, 9.0, 4.0],
    'form_efficiency': [1.09, 0.78, 1.00],
    'barrier_advantage': [5.5, 0.5, 3.5]
})

# Preprocess the new data
new_data_imputed = pd.DataFrame(imputer.transform(new_data), columns=new_data.columns)
new_data_scaled = pd.DataFrame(scaler.transform(new_data_imputed), columns=new_data_imputed.columns)

# Make predictions
new_pos_pred = gb_regressor.predict(new_data_scaled)
new_bin_pred = gb_classifier.predict(new_data_scaled)
new_bin_proba = gb_classifier.predict_proba(new_data_scaled)[:, 1]

# Create results dataframe
results_df = pd.DataFrame({
    'Horse_ID': [f'H{i+1}' for i in range(len(new_data))],
    'Predicted_Position': np.round(new_pos_pred, 2),
    'Top_3_Probability': np.round(new_bin_proba, 3),
    'Predicted_Top_3': ['Yes' if p == 1 else 'No' for p in new_bin_pred]
})

print("PREDICTIONS FOR NEW RACE:")
print(results_df)

print("\nRanked by Top 3 Probability (Highest First):")
ranked_results = results_df.sort_values('Top_3_Probability', ascending=False)
print(ranked_results)

## Summary and Conclusions

Recap of the model performance and findings

In [ ]:
print("FORM-TO-RESULTS ANALYSIS SUMMARY")
print("="*50)
print(f"Data Shape: {merged_data.shape}")
print(f"Features Used: {len(feature_columns)}")
print()
print("REGRESSION PERFORMANCE (Position Prediction):")
print(f"  - MSE: {mse:.3f}")
print(f"  - RMSE: {rmse:.3f}")
print(f"  - MAE: {mae:.3f}")
print(f"  - R²: {r2:.3f}")
print()
print("CLASSIFICATION PERFORMANCE (Top-3 Prediction):")
print(f"  - Accuracy: {acc:.3f}")
print()
print("CROSS-VALIDATION PERFORMANCE:")
print(f"  - Avg RMSE (Reg): {avg_rmse:.3f}")
print(f"  - Avg R² (Reg): {avg_r2:.3f}")
print(f"  - Avg Acc (Clf): {avg_acc:.3f}")
print()
print("KEY FINDINGS:")
print(f"  - Top feature for position prediction: {feature_importance_reg.iloc[0]['feature']}")
print(f"  - Top feature for top-3 prediction: {feature_importance_clf.iloc[0]['feature']}")
print(f"  - Model explains {(r2*100):.1f}% of variance in finishing positions")
print()
print("USAGE INSTRUCTIONS:")
print("  1. Prepare new race data with the same features")
print("  2. Apply the same preprocessing steps (imputation, scaling)")
print("  3. Use the trained models to predict positions and top-3 probabilities")
print("  4. Rank horses by predicted probabilities for betting decisions")